In [1]:
import pandas as pd
import numpy as np
import tensorflow as tf
#import tensorflow_addons as tfa
from tensorflow.keras.datasets import imdb
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Flatten
from tensorflow.keras.layers import LSTM,Bidirectional,GRU
from tensorflow.keras.layers import Embedding
from tensorflow.keras.preprocessing import sequence
from tensorflow.keras.utils import to_categorical
import datetime
import io
import itertools
# import seaborn as sns


2024-07-11 17:23:23.041794: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2024-07-11 17:23:23.042027: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2024-07-11 17:23:23.070161: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2024-07-11 17:23:23.167914: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2024-07-11 17:23:24.728967: W tensorflow/compiler/tf2

In [2]:
print(tf.config.list_physical_devices('GPU'))

[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]


2024-07-11 17:23:26.119886: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-07-11 17:23:26.558276: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-07-11 17:23:26.558423: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.


In [3]:
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler, StandardScaler
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import train_test_split

from sklearn.model_selection import KFold
from sklearn.metrics import classification_report

import sys
import os
# Obtener la ruta del directorio actual
os.chdir('/home/rgadea/nuevas_investigaciones_alimentos_2024')
current_dir = os.getcwd()
print(current_dir)

# Construir la ruta relativa al directorio que quieres agregar
relative_dir = os.path.join(current_dir, 'mis_pkgs/')

# Agregar la ruta relativa al sys.path
sys.path.insert(0, relative_dir)

#from MIOPATIA_db import DB_management as db 


/home/rgadea/nuevas_investigaciones_alimentos_2024


In [4]:
numero_muestras=201
numero_clases=2
entrada=[5,6]
numero_entradas =2
numero_epochs=20000

Voy a quedarme con los 50 atunes P1 para obtener conjunto de training y validacion

In [5]:
filename = "COPIA_PANDAS/medidas_agilent_2023_y_2024_201_puntos_clasificados.hdf"
with pd.HDFStore(filename,complib="zlib",complevel=4) as hdf_db:
    pre_p_e1  = hdf_db.get('data/pollos_estado')
    pre_p_e1 = pre_p_e1.loc[pre_p_e1['Pollo'] != 0]
    # p_e =pre_p_e1.drop_duplicates(subset = ['Pollo', 'Medida'],  keep = 'last').reset_index(drop = True)
    t    = hdf_db.get('data/tabla')
    X_train=np.zeros((pre_p_e1.shape[0],numero_muestras,numero_entradas))
    y_train=np.zeros((pre_p_e1.shape[0],1))
    x=0
    for index, row in pre_p_e1.iterrows():   # El primer registro no se toma en cuenta porque es basura
        Primero = int(row['Primero'])
        Ultimo  = int(row['Ultimo'])
        estado  = int(row['Estado'])
        #print(Primero)
        #print(Ultimo)
        #print(estado)
        if numero_clases==2:
            if estado == 0 or estado== 1:
                target = 0
            else:
                target = 1
        else:
            target=estado
        pepito=np.array(t.iloc[Primero:Ultimo+1])
        # #print(pepito.shape)
        X_train[x,:, 0] = pepito[:, 2] * pepito[:, 5]
        X_train[x,:, 1] = pepito[:, 2] * pepito[:, 6]

        #print(X_train[x][0:4,:])       
        y_train[x]=target
        y_train_to_categorical = to_categorical(y_train)
        x=x+1



X_train_filtrado = X_train
#y_train_filtrado = y_train
y_train_filtrado = y_train_to_categorical


scaler = MinMaxScaler(feature_range=(0, 1))
#scaler = StandardScaler()



#data1=np.concatenate((X_train_filtrado,X_test_filtrado1),axis=0) 

data_2d = X_train_filtrado.reshape(-1, X_train_filtrado.shape[-1])
normalized_data_2d = scaler.fit_transform(data_2d)



X_train_Normalizado=normalized_data_2d.reshape(X_train_filtrado.shape)
y_train_Normalizado=y_train_filtrado # los valores ya estaban normalizados
print(data_2d.shape)
print(X_train_Normalizado.shape)
print(y_train_Normalizado.shape)

inputs=X_train_Normalizado.reshape(X_train_Normalizado.shape[0],-1)
targets=y_train_Normalizado

print(inputs.shape)
print(targets.shape)



(38994, 2)
(194, 201, 2)
(194, 2)
(194, 402)
(194, 2)


Vamos a hacer los conjuntos de entrenamiento validacion y test

In [6]:
factor_aprendizaje=0.001
dimension_LSTM=200
dimension_dense1=50
dimension_dense2=20
algoritmo='rmsprop'
supermax=8*4
lossfunction='categorical_crossentropy'
def create_model():

    model = Sequential()
    model.add(Bidirectional(GRU(dimension_LSTM, return_sequences=True,recurrent_regularizer='L2',input_shape=(numero_muestras, numero_entradas))))
    model.add(Flatten())  
    #model.add(GRU(50, return_sequences=True))
    #model.add(GRU(50, return_sequences=False, recurrent_regularizer='L2'))
    model.add(Dense(dimension_dense1, activation='tanh', activity_regularizer='L2'))
    model.add(Dense(dimension_dense2, activation='tanh'))
    model.add(Dense(numero_clases, activation='softmax'))
    model.compile(loss=lossfunction, optimizer=algoritmo, metrics=['accuracy',
                              tf.keras.metrics.Recall(class_id=0),
                              tf.keras.metrics.Recall(class_id=1) #,
                              #tfa.metrics.F1Score(num_classes=numero_clases,average='macro', threshold=0.5)
                              ])
    model.optimizer.lr=(factor_aprendizaje)
    return model



In [7]:

experimento="LOMOS_Agilent_5clases_GRU1_{}_dense1_{}_dense2_{}_loss_{}_lr_{}_algoritmo_{}".format(dimension_LSTM,dimension_dense1,dimension_dense2,lossfunction,factor_aprendizaje,algoritmo)
logdir="./logs/defs/leavekout/{}_{}".format(experimento,datetime.datetime.now().strftime("%Y%m%d-%H%M%S"))
tensorboard_callback=tf.keras.callbacks.TensorBoard(log_dir=logdir, histogram_freq=1)
file_writer_cm = tf.summary.create_file_writer(logdir + '/cm')


2024-07-11 17:23:29.598861: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-07-11 17:23:29.598973: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-07-11 17:23:29.599012: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-07-11 17:23:29.974048: I external/local_xla/xla/stream_executor/cuda/cuda_executor.cc:887] could not open file to read NUMA node: /sys/bus/pci/devices/0000:01:00.0/numa_node
Your kernel may have been built without NUMA support.
2024-07-11 17:23:29.974240: I external/local_xla/xla/stream_executor

In [8]:
if numero_clases==2:
    class_names=['Buenos', 'Malos']
else:
    class_names=['A', 'B+', 'B', 'B-','C']

In [9]:
lr_callback = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_accuracy',
    factor=0.95,
    patience=1000,
    min_lr=0.0001
)
early_stop=tf.keras.callbacks.EarlyStopping(monitor='val_accuracy', min_delta=0, patience=2000, verbose=2, mode='auto', baseline=None, restore_best_weights=True)
# Define the K-fold Cross Validator
kfold = KFold(n_splits=10, shuffle=True)
# Define per-fold score containers
acc_per_fold = []
loss_per_fold = []
sensibilidad_YAKE_per_fold=[]
sensibilidad_no_YAKE_per_fold=[]
# K-fold Cross Validation model evaluation
fold_no = 1
for train, test in kfold.split(inputs, targets):
    model=create_model()
     # Generate a print
    print('------------------------------------------------------------------------')
    print(f'Training for fold {fold_no} ...')
    inputs_good=inputs.reshape(X_train_filtrado.shape)
    # Fit data to model
    history = model.fit(inputs_good[train], targets[train],
              batch_size=20,
              epochs=numero_epochs,
              callbacks=[early_stop,lr_callback, tensorboard_callback],
              validation_data=(inputs_good[test],targets[test])
              )
    if numero_clases==2:
        target_names = ['Buenos', 'Malos']
    else:   
        target_names = ['A', 'B+', 'B', 'B-','C']
    y_pred = model.predict(inputs_good[test])
    y_pred2=np.argmax(y_pred,axis=1)
    y_test_def2=np.argmax(targets[test],axis=1)
    print(classification_report(y_test_def2, y_pred2, target_names=target_names, digits=4))
    # Generate generalization metrics
    scores = model.evaluate(inputs_good[test], targets[test], verbose=0)
    print(f'Score for fold {fold_no}: {model.metrics_names[0]} of {scores[0]}; {model.metrics_names[1]} of {scores[1]*100}%')
    print(scores[2])
    print(scores[3])
   # print(scores[4])
    acc_per_fold.append(scores[1] * 100)
    loss_per_fold.append(scores[0])
    sensibilidad_YAKE_per_fold.append(scores[2] * 100)
    sensibilidad_no_YAKE_per_fold.append(scores[3] * 100)
    # Increase fold number
    fold_no = fold_no + 1

------------------------------------------------------------------------
Training for fold 1 ...


2024-07-11 17:23:35.248143: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


Epoch 1/20000


2024-07-11 17:23:41.069972: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8904
2024-07-11 17:23:41.228610: I external/local_xla/xla/service/service.cc:168] XLA service 0x7f70f50d34f0 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2024-07-11 17:23:41.228657: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA GeForce GTX 1070 Ti, Compute Capability 6.1
2024-07-11 17:23:41.265287: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1720711421.392054 1222163 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


9/9 [==============================] - ETA: 0s - loss: 4.1675 - accuracy: 0.5230 - recall: 0.4646 - recall_1: 0.6000

...

9/9 [==============================] - 1s 157ms/step - loss: 2.1672 - accuracy: 0.5287 - recall: 0.6970 - recall_1: 0.3067 - val_loss: 1.8753 - val_accuracy: 0.5000 - val_recall: 0.3000 - val_recall_1: 0.7000 - lr: 0.0010
Epoch 4/20000
9/9 [==============================] - ETA: 0s - loss: 1.7053 - accuracy: 0.5230 - recall: 0.7576 - recall_1: 0.2133

2024-07-11 17:23:47.783418: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 964800000 exceeds 10% of free system memory.


9/9 [==============================] - 1s 148ms/step - loss: 1.7053 - accuracy: 0.5230 - recall: 0.7576 - recall_1: 0.2133 - val_loss: 1.4991 - val_accuracy: 0.4500 - val_recall: 0.5000 - val_recall_1: 0.4000 - lr: 0.0010
Epoch 5/20000
9/9 [==============================] - ETA: 0s - loss: 1.3673 - accuracy: 0.5920 - recall: 0.8990 - recall_1: 0.1867

2024-07-11 17:23:49.141213: W external/local_tsl/tsl/framework/cpu_allocator_impl.cc:83] Allocation of 964800000 exceeds 10% of free system memory.


9/9 [==============================] - 1s 167ms/step - loss: 1.3673 - accuracy: 0.5920 - recall: 0.8990 - recall_1: 0.1867 - val_loss: 1.2390 - val_accuracy: 0.4500 - val_recall: 0.0000e+00 - val_recall_1: 0.9000 - lr: 0.0010
Epoch 6/20000
9/9 [==============================] - 1s 159ms/step - loss: 1.1811 - accuracy: 0.4655 - recall: 0.5455 - recall_1: 0.3600 - val_loss: 1.0438 - val_accuracy: 0.6500 - val_recall: 0.8000 - val_recall_1: 0.5000 - lr: 0.0010
Epoch 7/20000
9/9 [==============================] - 1s 139ms/step - loss: 0.9734 - accuracy: 0.5805 - recall: 0.6465 - recall_1: 0.4933 - val_loss: 1.0752 - val_accuracy: 0.5000 - val_recall: 1.0000 - val_recall_1: 0.0000e+00 - lr: 0.0010
Epoch 8/20000
9/9 [==============================] - 1s 153ms/step - loss: 0.8823 - accuracy: 0.5287 - recall: 0.8485 - recall_1: 0.1067 - val_loss: 0.8388 - val_accuracy: 0.5500 - val_recall: 1.0000 - val_recall_1: 0.1000 - lr: 0.0010
Epoch 9/20000
9/9 [==============================] - 1s 155ms/

In [ ]:
# == Provide average scores ==
print('------------------------------------------------------------------------')
print('Score per fold')
for i in range(0, len(acc_per_fold)):
  print('------------------------------------------------------------------------')
  print(f'> Fold {i+1} - Loss: {loss_per_fold[i]} - Accuracy: {acc_per_fold[i]}%')
print('------------------------------------------------------------------------')
print('Average scores for all folds:')
print(f'> Accuracy: {np.mean(acc_per_fold)} (+- {np.std(acc_per_fold)})')
print(f'> Loss: {np.mean(loss_per_fold)}')
print('------------------------------------------------------------------------')

------------------------------------------------------------------------
Score per fold
------------------------------------------------------------------------
> Fold 1 - Loss: 0.6678581237792969 - Accuracy: 75.0%
------------------------------------------------------------------------
> Fold 2 - Loss: 3.2928223609924316 - Accuracy: 44.999998807907104%
------------------------------------------------------------------------
> Fold 3 - Loss: 0.7720484733581543 - Accuracy: 80.0000011920929%
------------------------------------------------------------------------
> Fold 4 - Loss: 3.3535072803497314 - Accuracy: 40.00000059604645%
------------------------------------------------------------------------
> Fold 5 - Loss: 3.251016855239868 - Accuracy: 57.894736528396606%
------------------------------------------------------------------------
> Fold 6 - Loss: 3.4764716625213623 - Accuracy: 31.578946113586426%
------------------------------------------------------------------------
> Fold 7 - 

In [ ]:
# == Provide average scores ==
print('------------------------------------------------------------------------')
print('Score per fold')
for i in range(0, len(sensibilidad_YAKE_per_fold)):
  print('------------------------------------------------------------------------')
  print(f'> Fold {i+1} - Sensibilidad YAKE: {sensibilidad_YAKE_per_fold[i]} - Sensibilidad NOYAKE: {sensibilidad_no_YAKE_per_fold[i]}%')
print('------------------------------------------------------------------------')
print('Average scores for all folds:')
print(f'> Sensibilidad YAKE: {np.mean(sensibilidad_YAKE_per_fold)} (+- {np.std(sensibilidad_YAKE_per_fold)})')
print(f'> Sensibilidad NOYAKE: {np.mean(sensibilidad_no_YAKE_per_fold)} (+- {np.std(sensibilidad_no_YAKE_per_fold)})')
print('------------------------------------------------------------------------')

------------------------------------------------------------------------
Score per fold
------------------------------------------------------------------------
> Fold 1 - Sensibilidad YAKE: 100.0 - Sensibilidad NOYAKE: 28.57142984867096%
------------------------------------------------------------------------
> Fold 2 - Sensibilidad YAKE: 88.88888955116272 - Sensibilidad NOYAKE: 9.090909361839294%
------------------------------------------------------------------------
> Fold 3 - Sensibilidad YAKE: 100.0 - Sensibilidad NOYAKE: 33.33333432674408%
------------------------------------------------------------------------
> Fold 4 - Sensibilidad YAKE: 9.090909361839294 - Sensibilidad NOYAKE: 77.77777910232544%
------------------------------------------------------------------------
> Fold 5 - Sensibilidad YAKE: 100.0 - Sensibilidad NOYAKE: 0.0%
------------------------------------------------------------------------
> Fold 6 - Sensibilidad YAKE: 0.0 - Sensibilidad NOYAKE: 100.0%
----------